In [12]:
import scanpy as sc
import anndata as ad
import h5py
from anndata._io.specs import read_elem
import pandas as pd
import numpy as np
import re
from os import makedirs

def load_anndata(path: str) -> ad.AnnData:
    """
    Load a test dataset for unit testing purposes.
    
    Returns:
        ad.AnnData: An AnnData object containing the test dataset.
    """
    # Load the example dataset from Scanpy
    adata = sc.read_h5ad(path, backed='r')
    # Return the AnnData object
    return adata


def load_metadata(path: str) -> pd.DataFrame:
    """
    Read and load only metadata  (obs) for testing purposes.
    
    Returns:
        pd.DataFrame
    """
    with h5py.File(path) as f:
        obs = read_elem(f["obs"])
    metadata = obs.drop_duplicates(subset=["donor_id"], keep="first")
    return metadata

def save_matrix_subclass_patient(donor : str, path : str, genes : list, dir_path : str) -> None:
    """
    Read and load matrix for each patient in adata
    Args:
        donor: Donor ID.
        path: adata path.
    """
    adata = load_anndata(path)
    match = re.search(f"./(.*?)/(.*?).h5ad", path)
    if match:
        dir = match.group(1)
        subclass = match.group(2)
    adata = adata[adata.obs["donor_id"] == donor]
    df = adata.to_df()[genes].T
    donor = donor.replace(".", "-")
    df.to_csv(f"{dir_path}matrix/{donor}_{subclass}.tsv", sep="\t")
    del adata, df

def filter_subtype(path: str, subclass: str, dir_path : str) -> ad.AnnData:
    adata = load_anndata(path)
    adata.obs = adata.obs[["donor_id", "Subclass"]]
    adata_filtered = adata[adata.obs["Subclass"] == subclass]
    subclass = subclass.replace(" ", "-").replace("/", "-")
    output = f"{dir_path}{subclass}.h5ad"
    adata_filtered.write_h5ad(output)
    del adata_filtered
    return output



In [15]:
import sys
import os

import pandas as pd
from joblib import Parallel, delayed
import glob
import subprocess
import anndata as ad
import re
import polars as pl

def main():
    n_threads = 25
    print("Starting Python prediction pipeline...")
    
    input_dir = "/export/space3/users/rafadiaz/NeuroNet_AD"
    output_dir = "/export/space3/users/rafadiaz/NeuroNet_AD/output"
    data_dir = "/export/space3/users/rafadiaz/NeuroNet_AD/app/data/"
    
    if not os.path.exists(input_dir):
        print(f"Warning: Input directory not found: {input_dir}")
    os.makedirs(output_dir, exist_ok=True)
    print(f"Input directory: {input_dir}")
    print(f"Output directory: {output_dir}")

    path = "/export/space3/users/rafadiaz/NeuroNet_AD/92b37feb-aa2c-40d7-bd90-0a9b5ddb3b27.h5ad"
    
    #Obtain metadata and filter duplicates
    metadata = load_metadata(path)
    donors = metadata["donor_id"].unique().tolist()
    metadata.to_csv(f"{data_dir}metadata_pre.csv", index=False)
    del metadata
    df_subclass = pd.read_csv(f"{data_dir}filtered_links.csv")
    list_subclass = df_subclass["Subclass"].unique().tolist()
    os.makedirs(f"{data_dir}matrix", exist_ok=True)
    for subclass in list_subclass:
        df_sub = df_subclass[df_subclass["Subclass"] == subclass]
        df_genes = df_sub["Gene"]
        list_genes = df_genes.tolist()
        adata_path = filter_subtype(path, subclass, data_dir)
        subclass = subclass.replace(" ", "-").replace("/", "-")
        df_genes.to_csv(f"{data_dir}genes_{subclass}.txt", index=False, header=False)
        Parallel(n_jobs=6)(
            delayed(save_matrix_subclass_patient)(donor, adata_path, list_genes, data_dir)
            for donor in donors
        )
        
main()

Starting Python prediction pipeline...
Input directory: /export/space3/users/rafadiaz/NeuroNet_AD
Output directory: /export/space3/users/rafadiaz/NeuroNet_AD/output


FileNotFoundError: [Errno 2] No such file or directory: '/export/space3/users/rafadiaz/NeuroNet_AD/app/data/filtered_links.csv'

In [ ]:
import anndata as ad
import scanpy as sc

ruta = "/export/space3/users/rafadiaz/NeuroNet_AD/92b37feb-aa2c-40d7-bd90-0a9b5ddb3b27.h5ad"

adata = ad.read_h5ad(ruta, backed="r")
subclasses = adata.obs["Subclass"].cat.categories.tolist()

print(len(subclasses))
for s in subclasses:
    print(s)


exp_component_name
GGTGATTAGGTCACTT-L8TX_210722_01_H06-1153814299    Oligodendrocyte
TTGAACGCAGGTGTGA-L8TX_210729_01_G12-1153814338              L5 IT
GGGAGTAAGGCATTTC-L8TX_210107_01_H09-1142430361            L2/3 IT
ACAGAAAGTATCGTGT-L8TX_210415_01_G01-1153814188            L5/6 NP
TTGTTCAAGCGAGAAA-L8TX_210513_01_F11-1153814259    Oligodendrocyte
                                                       ...       
GATGCTATCAAGCTTG-L8TX_210401_01_B09-1153814174            L2/3 IT
ATGGTTGAGGTCTACT-L8TX_210722_01_D08-1153814330            L2/3 IT
GTCGAATCAACAAGTA-L8TX_201016_01_E05-1153814151    Oligodendrocyte
CCTGCATCATGATAGA-L8TX_210408_01_E10-1153814184              L4 IT
AGGATCTGTCCTCATC-L8TX_201030_01_B12-1142430231         L6 IT Car3
Name: Subclass, Length: 1378211, dtype: category
Categories (24, object): ['Lamp5 Lhx6', 'Lamp5', 'Pax6', 'Sncg', ..., 'Oligodendrocyte', 'Endothelial', 'VLMC', 'Microglia-PVM']

In [ ]:
import pandas as pd
import h5py
from anndata.experimental import read_elem
def load_patients(adata) -> pd.DataFrame:
    """
    Read and load only the patients.
    
    Returns:
        pd.DataFrame
    """
    with h5py.File(adata.filename) as f:
        patients_subclass = read_elem(f["obs"])[["donor_id", "Subclass"]].drop_duplicates()
    return patients_subclass

patients_subclass=load_patients(adata)

patients_subclass.to_csv("/export/space3/users/rafadiaz/NeuroNet_AD/output/patients_subclass.csv", index=False)


/export/space3/users/rafadiaz/conda_envs/neuronet/lib/python3.12/site-packages/anndata/experimental/__init__.py:48: FutureWarning: Importing read_elem from `anndata.experimental` is deprecated. Import anndata.io.read_elem instead.
  return module_get_attr_redirect(


OSError: Cannot save file into a non-existent directory: '/output'

In [6]:
import pandas as pd

df = pd.read_csv(
    "/export/space3/users/rafadiaz/NeuroNet_AD/app/data/matrix/H20-33-001_Vip.tsv",
    sep="\t",
    index_col=0
)

print(df.shape)
print(df.head())

(35483, 1422)
        ATCGTAGTCCTACGAA-L8TX_210430_01_C04-1142430413  \
TSPAN6                                             0.0   
TNMD                                               0.0   
DPM1                                               0.0   
SCYL3                                              0.0   
FIRRM                                              0.0   

        ATCACAGGTGTGTGGA-L8TX_210430_01_C04-1142430413  \
TSPAN6                                        0.000000   
TNMD                                          0.000000   
DPM1                                          0.778787   
SCYL3                                         0.000000   
FIRRM                                         0.000000   

        TCGGTCTAGGTCGCCT-L8TX_210430_01_C04-1142430413  \
TSPAN6                                        0.000000   
TNMD                                          0.000000   
DPM1                                          1.224793   
SCYL3                                         0.000000  

In [ ]:
path = "/export/space3/users/rafadiaz/NeuroNet_AD/92b37feb-aa2c-40d7-bd90-0a9b5ddb3b27.h5ad"

adata = ad.read_h5ad(path, backed="r")
print(adata.var_names[:10])       # ¿cómo se llaman los genes?
print(adata.var.columns.tolist()) # ¿hay una columna con el symbol?

NameError: name 'ad' is not defined